# Ninapro DB2 — EMG 제스처 분류 베이스 파이프라인

**Dataset**: Ninapro DB2  
- 피험자: 40명  
- 운동 클래스: 49가지 손 동작 + Rest (총 50 클래스)  
- EMG 전극: 12채널 (Delsys Trigno 무선 센서)  
- 샘플링 레이트: 2000 Hz  
- 가속도계: 36채널 (12전극 × 3축)

### 파이프라인 구성
```
1. 환경 설치
2. 데이터 로딩 (.mat)
3. 전처리 (필터링 → 윈도잉 → 정규화)
4. 특징 추출 (시계열 / 주파수 도메인)
5. 모델 학습 (CNN-1D, LSTM, SVM)
6. 평가 및 시각화
```

---
## 0. 환경 설치

In [ ]:
# 필요한 패키지 설치 (최초 1회)
import subprocess, sys

pkgs = [
    "numpy", "scipy", "pandas", "matplotlib", "seaborn",
    "scikit-learn", "torch", "torchinfo",
    "tqdm", "h5py", "PyWavelets"
]

for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ 패키지 설치 완료")

---
## 1. 데이터 구조 확인 및 로딩

### 폴더 구조 예시
```
data/
  S1_E1_A1.mat   ← 피험자1, 운동세트1 (동작 1~17)
  S1_E2_A1.mat   ← 피험자1, 운동세트2 (동작 18~40)
  S1_E3_A1.mat   ← 피험자1, 운동세트3 (동작 41~49)
  S2_E1_A1.mat
  ...
```

### .mat 파일 주요 키
| 키 | 설명 |
|---|---|
| `emg` | EMG 신호 (N × 12) |
| `stimulus` | 운동 레이블 (N,) — 0=rest |
| `restimulus` | 재분류된 레이블 |
| `repetition` | 반복 번호 (1~6) |
| `rerepetition` | 재분류된 반복 번호 |

In [ ]:
import os
import glob
import numpy as np
import scipy.io as sio
import h5py
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

# ─── 설정 ────────────────────────────────────────────────────────────────────
DATA_DIR   = "./data"          # .mat 파일이 있는 폴더 경로
SUBJECTS   = list(range(1, 41))  # 사용할 피험자 번호 (1~40)
EXERCISES  = [1, 2, 3]          # 운동 세트 (E1~E3)
FS         = 2000               # 샘플링 레이트 (Hz)
N_CHANNELS = 12                 # EMG 채널 수
# ─────────────────────────────────────────────────────────────────────────────


def load_mat(filepath: str) -> dict:
    """scipy 또는 h5py로 .mat 파일 로드 (v7.3 대응)"""
    try:
        data = sio.loadmat(filepath, squeeze_me=True)
        return data
    except NotImplementedError:
        # MATLAB v7.3 HDF5 형식
        data = {}
        with h5py.File(filepath, 'r') as f:
            for key in ['emg', 'stimulus', 'restimulus', 'repetition', 'rerepetition']:
                if key in f:
                    data[key] = np.array(f[key]).T  # HDF5는 전치 필요
        return data


def load_subject(subject_id: int, data_dir: str = DATA_DIR,
                 exercises: list = EXERCISES,
                 use_restimulus: bool = True) -> dict:
    """
    한 피험자의 모든 운동 세트를 로드 후 병합.
    use_restimulus=True → 재분류된 레이블 사용 (권장)
    """
    all_emg, all_labels, all_reps = [], [], []

    # 운동 세트별 레이블 오프셋 (DB2 기준)
    label_offsets = {1: 0, 2: 17, 3: 40}

    for ex in exercises:
        pattern = os.path.join(data_dir, f"S{subject_id}_E{ex}_A1.mat")
        files = glob.glob(pattern)
        if not files:
            print(f"  [!] 파일 없음: {pattern}")
            continue

        mat = load_mat(files[0])
        emg   = mat['emg'].astype(np.float32)           # (N, 12)
        label_key = 'restimulus' if use_restimulus and 'restimulus' in mat else 'stimulus'
        rep_key   = 'rerepetition' if use_restimulus and 'rerepetition' in mat else 'repetition'

        labels = mat[label_key].astype(np.int32).ravel()
        reps   = mat[rep_key].astype(np.int32).ravel()

        # 레이블 오프셋 적용 (rest=0은 유지)
        offset = label_offsets[ex]
        labels_shifted = np.where(labels > 0, labels + offset, 0)

        all_emg.append(emg)
        all_labels.append(labels_shifted)
        all_reps.append(reps)

    if not all_emg:
        raise FileNotFoundError(f"Subject {subject_id} 데이터 없음")

    return {
        'emg':    np.concatenate(all_emg,    axis=0),
        'labels': np.concatenate(all_labels, axis=0),
        'reps':   np.concatenate(all_reps,   axis=0),
    }


# ─── 단일 피험자 로드 테스트 ──────────────────────────────────────────────────
print("데이터 로딩 중...")
subject_id = 1  # 테스트용

if os.path.exists(DATA_DIR):
    data = load_subject(subject_id)
    emg, labels, reps = data['emg'], data['labels'], data['reps']
    print(f"S{subject_id} — EMG shape: {emg.shape}, 레이블 분포:")
    unique, counts = np.unique(labels, return_counts=True)
    print(pd.Series(counts, index=unique).head(10))
else:
    print(f"[!] DATA_DIR='{DATA_DIR}' 없음. 더미 데이터로 파이프라인 테스트합니다.")
    # ─ 더미 데이터 생성 (실제 데이터 없을 때 파이프라인 확인용) ─
    N_SAMPLES  = 200_000
    N_CLASSES  = 50
    emg    = np.random.randn(N_SAMPLES, N_CHANNELS).astype(np.float32) * 0.001
    labels = np.zeros(N_SAMPLES, dtype=np.int32)
    # 블록 단위로 레이블 할당
    block  = N_SAMPLES // (N_CLASSES * 2)
    for i in range(N_CLASSES):
        s = i * block * 2 + block // 2
        e = s + block
        if e < N_SAMPLES:
            labels[s:e] = i
    reps   = np.repeat(np.arange(1, 7), N_SAMPLES // 6 + 1)[:N_SAMPLES]
    print(f"더미 데이터 — EMG: {emg.shape}, 클래스 수: {len(np.unique(labels))}")

print("\n✅ 데이터 로딩 완료")

---
## 2. 신호 시각화

In [ ]:
def plot_emg_overview(emg, labels, fs=FS, duration_sec=5, start_sec=0):
    """EMG 신호 및 레이블 오버뷰 플롯"""
    s = int(start_sec * fs)
    e = int((start_sec + duration_sec) * fs)
    e = min(e, len(emg))
    t = np.arange(e - s) / fs

    fig, axes = plt.subplots(N_CHANNELS + 1, 1, figsize=(14, 14), sharex=True)
    fig.suptitle(f"EMG 신호 ({duration_sec}초, {N_CHANNELS}채널)", fontsize=14, fontweight='bold')

    # 레이블
    axes[0].fill_between(t, labels[s:e], alpha=0.6, color='steelblue')
    axes[0].set_ylabel("레이블", fontsize=8)
    axes[0].set_ylim(-1, labels.max() + 1)

    # 각 채널
    colors = plt.cm.tab20(np.linspace(0, 1, N_CHANNELS))
    for ch in range(N_CHANNELS):
        ax = axes[ch + 1]
        ax.plot(t, emg[s:e, ch], lw=0.5, color=colors[ch])
        ax.set_ylabel(f"CH{ch+1}", fontsize=8)
        ax.set_yticks([])

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

plot_emg_overview(emg, labels, start_sec=0)

---
## 3. 전처리

### 처리 순서
1. **대역통과 필터** (20–500 Hz) — 노이즈 및 DC 제거  
2. **노치 필터** (50 Hz) — 전원 노이즈 제거  
3. **윈도잉** — 슬라이딩 윈도우 (200ms, 100ms 오버랩)  
4. **정규화** — 채널별 Z-score

In [ ]:
from scipy.signal import butter, filtfilt, iirnotch

# ─── 필터 파라미터 ────────────────────────────────────────────────────────────
BP_LOW, BP_HIGH = 20, 500   # 대역통과 (Hz)
NOTCH_FREQ      = 50        # 노치 (Hz)
NOTCH_Q         = 30        # 노치 품질 계수
BUTTER_ORDER    = 4
# ─────────────────────────────────────────────────────────────────────────────

def bandpass_filter(signal, lowcut=BP_LOW, highcut=BP_HIGH, fs=FS, order=BUTTER_ORDER):
    nyq = fs / 2
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, signal, axis=0)

def notch_filter(signal, freq=NOTCH_FREQ, fs=FS, Q=NOTCH_Q):
    b, a = iirnotch(freq / (fs / 2), Q)
    return filtfilt(b, a, signal, axis=0)

def preprocess_emg(emg_raw: np.ndarray) -> np.ndarray:
    """EMG 신호 전처리 (대역통과 + 노치 필터)"""
    emg_bp    = bandpass_filter(emg_raw)
    emg_clean = notch_filter(emg_bp)
    return emg_clean.astype(np.float32)

print("전처리 중...")
emg_filtered = preprocess_emg(emg)
print(f"✅ 필터링 완료: {emg_filtered.shape}")

# 필터 전/후 비교
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
t_plot = np.arange(4000) / FS
axes[0].plot(t_plot, emg[5000:9000, 0], lw=0.7, color='gray', label='Raw')
axes[1].plot(t_plot, emg_filtered[5000:9000, 0], lw=0.7, color='royalblue', label='Filtered')
for ax in axes:
    ax.legend(loc='upper right'); ax.set_ylabel('Amplitude (V)')
axes[-1].set_xlabel('Time (s)')
plt.suptitle('EMG 필터링 비교 (CH 1)', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 4. 슬라이딩 윈도우 세그먼테이션

In [ ]:
# ─── 윈도우 파라미터 ──────────────────────────────────────────────────────────
WIN_MS    = 200     # 윈도우 길이 (ms)
STEP_MS   = 100     # 스텝 길이 (ms) — 50% 오버랩
WIN_LEN   = int(WIN_MS  / 1000 * FS)   # 샘플 수로 변환
STEP_LEN  = int(STEP_MS / 1000 * FS)
# Train: rep 1~4, Val: rep 5, Test: rep 6 (leave-one-rep-out)
# ─────────────────────────────────────────────────────────────────────────────

def segment_signal(emg: np.ndarray, labels: np.ndarray, reps: np.ndarray,
                   win_len: int = WIN_LEN, step_len: int = STEP_LEN,
                   exclude_rest: bool = False) -> tuple:
    """
    슬라이딩 윈도우로 세그먼트 추출.
    레이블은 윈도우 내 최빈값(majority vote)으로 결정.
    """
    X_list, y_list, r_list = [], [], []
    n = len(emg)

    for start in range(0, n - win_len + 1, step_len):
        end = start + win_len
        seg_label = labels[start:end]
        seg_rep   = reps[start:end]

        # 레이블 일관성: 윈도우 내 최빈값
        label = np.bincount(seg_label).argmax()
        rep   = np.bincount(seg_rep).argmax()

        if exclude_rest and label == 0:
            continue

        X_list.append(emg[start:end])  # (win_len, n_channels)
        y_list.append(label)
        r_list.append(rep)

    X = np.array(X_list, dtype=np.float32)   # (N, win_len, n_channels)
    y = np.array(y_list, dtype=np.int64)
    r = np.array(r_list, dtype=np.int64)
    return X, y, r


print(f"윈도우 크기: {WIN_LEN} samples ({WIN_MS}ms)")
print(f"스텝 크기:   {STEP_LEN} samples ({STEP_MS}ms)")
print("세그먼테이션 중...")

X_all, y_all, r_all = segment_signal(emg_filtered, labels, reps, exclude_rest=False)
print(f"✅ 세그먼트: {X_all.shape}  레이블 분포 (상위 10):")
unique, counts = np.unique(y_all, return_counts=True)
print(pd.Series(counts[:10], index=unique[:10]))

---
## 5. 데이터 분할 (Leave-One-Rep-Out)

반복(Repetition) 기준으로 Train / Val / Test 분리  
→ **시간적 오염** 없는 평가 보장

In [ ]:
def split_by_rep(X, y, r, test_reps=(6,), val_reps=(5,)):
    """반복(rep) 기반 Train/Val/Test 분할"""
    test_mask  = np.isin(r, test_reps)
    val_mask   = np.isin(r, val_reps)
    train_mask = ~(test_mask | val_mask)

    return (
        X[train_mask], y[train_mask], r[train_mask],
        X[val_mask],   y[val_mask],   r[val_mask],
        X[test_mask],  y[test_mask],  r[test_mask],
    )

X_train, y_train, r_train, \
X_val,   y_val,   r_val,   \
X_test,  y_test,  r_test   = split_by_rep(X_all, y_all, r_all)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# 클래스별 정규화 (채널별 z-score, train 통계 사용)
mean = X_train.mean(axis=(0, 1), keepdims=True)
std  = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

print("✅ 정규화 완료")

# 레이블 재맵핑 (연속 정수로)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

N_CLASSES = len(le.classes_)
print(f"클래스 수: {N_CLASSES}")

---
## 6. 특징 추출 (ML 모델용)

시계열 + 주파수 도메인 특징 조합

In [ ]:
from scipy.stats import skew, kurtosis
from scipy.signal import welch

def extract_features(X: np.ndarray, fs: int = FS) -> np.ndarray:
    """
    윈도우 배열 (N, T, C)에서 특징 추출.
    시계열: MAV, RMS, WL, ZC, SSC
    주파수: MNF, MDF, PKF
    반환: (N, n_features)
    """
    N, T, C = X.shape
    feats = []

    for i in tqdm(range(N), desc="특징 추출", leave=False):
        seg = X[i]   # (T, C)
        f_ch = []

        for c in range(C):
            x = seg[:, c]

            # ── 시계열 특징 ─────────────────────────────
            mav  = np.mean(np.abs(x))           # Mean Absolute Value
            rms  = np.sqrt(np.mean(x**2))       # Root Mean Square
            var  = np.var(x)                    # Variance
            wl   = np.sum(np.abs(np.diff(x)))   # Waveform Length
            zc   = np.sum(np.diff(np.sign(x)) != 0)  # Zero Crossing
            ssc  = np.sum(np.diff(np.sign(np.diff(x))) != 0)  # Slope Sign Change
            sk   = skew(x)                      # Skewness
            ku   = kurtosis(x)                  # Kurtosis

            # ── 주파수 특징 ─────────────────────────────
            freqs, psd = welch(x, fs=fs, nperseg=min(256, T))
            total_power = np.sum(psd)
            if total_power > 0:
                mnf = np.sum(freqs * psd) / total_power  # Mean Frequency
                cum = np.cumsum(psd)
                mdf = freqs[np.searchsorted(cum, total_power / 2)]  # Median Freq
                pkf = freqs[np.argmax(psd)]              # Peak Frequency
            else:
                mnf = mdf = pkf = 0.0

            f_ch.extend([mav, rms, var, wl, zc, ssc, sk, ku, mnf, mdf, pkf])

        feats.append(f_ch)

    return np.array(feats, dtype=np.float32)


print("특징 추출 중 (Train)...")
F_train = extract_features(X_train)
print("특징 추출 중 (Val)...")
F_val   = extract_features(X_val)
print("특징 추출 중 (Test)...")
F_test  = extract_features(X_test)

n_features = F_train.shape[1]
print(f"\n✅ 특징 벡터: {n_features}개 ({N_CHANNELS}채널 × {n_features // N_CHANNELS}특징)")
print(f"Train: {F_train.shape}, Val: {F_val.shape}, Test: {F_test.shape}")

---
## 7. 모델 A — SVM (Baseline)

특징 벡터 기반 SVM — 빠른 베이스라인

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 특징 정규화
scaler = StandardScaler()
F_train_sc = scaler.fit_transform(F_train)
F_val_sc   = scaler.transform(F_val)
F_test_sc  = scaler.transform(F_test)

print("SVM 학습 중...")
svm = SVC(kernel='rbf', C=10, gamma='scale', decision_function_shape='ovr', cache_size=2000)
svm.fit(F_train_sc, y_train_enc)

svm_val_pred  = svm.predict(F_val_sc)
svm_test_pred = svm.predict(F_test_sc)

print(f"\n{'='*45}")
print(f"  SVM — Val  Accuracy: {accuracy_score(y_val_enc,  svm_val_pred):.4f}")
print(f"  SVM — Test Accuracy: {accuracy_score(y_test_enc, svm_test_pred):.4f}")
print(f"{'='*45}")

---
## 8. 모델 B — 1D CNN

원시 EMG 세그먼트를 입력으로 받는 딥러닝 분류기

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"디바이스: {DEVICE}")


class EMGDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # X: (N, T, C) → (N, C, T) for 1D Conv
        self.X = torch.from_numpy(X.transpose(0, 2, 1))  # (N, C, T)
        self.y = torch.from_numpy(y)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class CNN1D(nn.Module):
    """1D CNN for EMG classification"""
    def __init__(self, n_channels: int = 12, win_len: int = WIN_LEN, n_classes: int = 50):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(n_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64), nn.ELU(),
            nn.MaxPool1d(2), nn.Dropout(0.2),

            # Block 2
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ELU(),
            nn.MaxPool1d(2), nn.Dropout(0.2),

            # Block 3
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ELU(),
            nn.MaxPool1d(2), nn.Dropout(0.2),

            # Block 4
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ELU(),
            nn.AdaptiveAvgPool1d(4),   # 출력 크기 고정
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4, 512), nn.ELU(), nn.Dropout(0.4),
            nn.Linear(512, 256),     nn.ELU(), nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):  # x: (B, C, T)
        return self.classifier(self.features(x))


# 데이터로더
BATCH_SIZE = 256
train_loader = DataLoader(EMGDataset(X_train, y_train_enc), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(EMGDataset(X_val,   y_val_enc),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(EMGDataset(X_test,  y_test_enc),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = CNN1D(n_channels=N_CHANNELS, win_len=WIN_LEN, n_classes=N_CLASSES).to(DEVICE)

# 파라미터 수 출력
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"CNN1D 파라미터 수: {total_params:,}")
print(model)

In [ ]:
# ─── 학습 설정 ────────────────────────────────────────────────────────────────
EPOCHS    = 50
LR        = 1e-3
PATIENCE  = 10    # Early stopping patience
# ─────────────────────────────────────────────────────────────────────────────

criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)


def run_epoch(model, loader, optimizer=None, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss   = criterion(logits, yb)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            total      += len(yb)

    return total_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc  = 0.0
patience_cnt  = 0
best_weights  = None

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7} | {'LR':>8}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, train=True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    lr_now = scheduler.get_last_lr()[0]
    print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>9.4f} | {vl_loss:>8.4f} | {vl_acc:>7.4f} | {lr_now:.2e}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n조기 종료 (epoch {epoch}). Best Val Acc: {best_val_acc:.4f}")
            break

# 최고 가중치 복원
model.load_state_dict(best_weights)
print(f"\n✅ 학습 완료. Best Val Acc: {best_val_acc:.4f}")

---
## 9. 모델 C — CNN-LSTM (선택적)

CNN으로 공간 특징 추출 후 LSTM으로 시계열 패턴 모델링

In [ ]:
class CNN_LSTM(nn.Module):
    def __init__(self, n_channels=12, win_len=WIN_LEN, n_classes=50,
                 lstm_hidden=128, lstm_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_channels, 64,  kernel_size=5, padding=2), nn.BatchNorm1d(64),  nn.ELU(), nn.MaxPool1d(2),
            nn.Conv1d(64,         128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ELU(), nn.MaxPool1d(2),
        )  # out: (B, 128, T//4)

        self.lstm = nn.LSTM(
            input_size=128, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            dropout=0.3, bidirectional=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 256), nn.ELU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):   # x: (B, C, T)
        out = self.cnn(x)               # (B, 128, T')
        out = out.permute(0, 2, 1)      # (B, T', 128)
        out, _ = self.lstm(out)         # (B, T', lstm_hidden*2)
        out = out[:, -1, :]             # 마지막 타임스텝
        return self.head(out)


model_lstm = CNN_LSTM(n_channels=N_CHANNELS, win_len=WIN_LEN, n_classes=N_CLASSES).to(DEVICE)
total_params_lstm = sum(p.numel() for p in model_lstm.parameters() if p.requires_grad)
print(f"CNN-LSTM 파라미터 수: {total_params_lstm:,}")
print("\n※ CNN-LSTM 학습은 run_epoch() 동일하게 적용 가능")

---
## 10. 평가 및 시각화

In [ ]:
# ─── 학습 곡선 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN1D 학습 곡선', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── 테스트 세트 평가 ─────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            preds.extend(logits.argmax(1).cpu().numpy())
            truths.extend(yb.numpy())
    return np.array(truths), np.array(preds)

y_true, y_pred = evaluate(model, test_loader)

test_acc = accuracy_score(y_true, y_pred)
print(f"\n{'='*50}")
print(f"  CNN1D Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"{'='*50}")
print("\n분류 리포트 (상위 클래스):")
print(classification_report(y_true, y_pred, zero_division=0)[:1200])

In [ ]:
# ─── 혼동 행렬 ────────────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm_norm, ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0, cbar_kws={'label': 'Recall'})
ax.set_title(f'Confusion Matrix (Normalized) — Test Acc: {test_acc:.4f}', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
ax.tick_params(axis='both', labelsize=6)
plt.tight_layout(); plt.show()

In [ ]:
# ─── 클래스별 F1 바 차트 ─────────────────────────────────────────────────────
from sklearn.metrics import f1_score
f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(f1_per_class))
bars = ax.bar(x, f1_per_class, color=plt.cm.RdYlGn(f1_per_class))
ax.axhline(f1_per_class.mean(), color='navy', linestyle='--', label=f'Mean F1={f1_per_class.mean():.3f}')
ax.set_xticks(x); ax.set_xticklabels(le.classes_, fontsize=7, rotation=90)
ax.set_xlabel('Class'); ax.set_ylabel('F1 Score')
ax.set_title('클래스별 F1 Score', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

---
## 11. 모델 저장 / 로드

In [ ]:
import json

SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# 모델 저장
checkpoint = {
    'model_state':  model.state_dict(),
    'model_config': {'n_channels': N_CHANNELS, 'win_len': WIN_LEN, 'n_classes': N_CLASSES},
    'norm_stats':   {'mean': mean.tolist(), 'std': std.tolist()},
    'label_classes': le.classes_.tolist(),
    'history':       history,
    'test_accuracy': float(test_acc),
}

torch.save(checkpoint, f"{SAVE_DIR}/cnn1d_best.pth")
print(f"✅ 모델 저장: {SAVE_DIR}/cnn1d_best.pth")


# ─── 로드 예시 ────────────────────────────────────────────────────────────────
def load_model(path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    cfg  = ckpt['model_config']
    m    = CNN1D(**cfg).to(DEVICE)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    return m, ckpt

model_loaded, ckpt_loaded = load_model(f"{SAVE_DIR}/cnn1d_best.pth")
print(f"모델 로드 완료 | 저장된 Test Acc: {ckpt_loaded['test_accuracy']:.4f}")

---
## 12. 실시간 추론 시뮬레이션

스트리밍 EMG 신호 → 슬라이딩 윈도우 → 실시간 예측

In [ ]:
class RealtimeClassifier:
    """실시간 EMG 스트리밍 분류기"""

    def __init__(self, model, win_len=WIN_LEN, step_len=STEP_LEN,
                 mean=mean, std=std, le=le, device=DEVICE,
                 vote_window=5):
        self.model    = model
        self.win_len  = win_len
        self.step_len = step_len
        self.mean     = mean
        self.std      = std
        self.le       = le
        self.device   = device
        self.buffer   = np.zeros((0, N_CHANNELS), dtype=np.float32)
        self.vote_buf = []   # 다수결 투표용 버퍼
        self.vote_win = vote_window

    def push(self, new_samples: np.ndarray) -> int | None:
        """새 샘플 (M, C) 추가. 윈도우 완성 시 클래스 반환"""
        self.buffer = np.vstack([self.buffer, new_samples])
        result = None

        while len(self.buffer) >= self.win_len:
            seg = self.buffer[:self.win_len]             # (T, C)
            seg = (seg - self.mean[0]) / self.std[0]    # 정규화
            x   = torch.from_numpy(seg.T[None]).to(self.device)  # (1, C, T)

            self.model.eval()
            with torch.no_grad():
                logit = self.model(x)
                pred  = logit.argmax(1).item()
                prob  = torch.softmax(logit, 1).max().item()

            self.vote_buf.append(pred)
            if len(self.vote_buf) > self.vote_win:
                self.vote_buf.pop(0)

            # 다수결 투표
            voted = np.bincount(self.vote_buf).argmax()
            result = (self.le.inverse_transform([voted])[0], prob)

            self.buffer = self.buffer[self.step_len:]    # 슬라이드

        return result


# 시뮬레이션
clf = RealtimeClassifier(model_loaded)
sim_emg = emg_filtered[10_000:14_000]   # 2초치 EMG

print("실시간 추론 시뮬레이션 (스텝별):\n")
chunk = 100  # 한 번에 push할 샘플 수
for i in range(0, len(sim_emg) - chunk, chunk):
    result = clf.push(sim_emg[i:i+chunk])
    if result:
        label_idx, prob = result
        print(f"  t={i/FS*1000:.0f}ms → 클래스 {label_idx:2d} (confidence: {prob:.3f})")

---
## 13. 다음 단계 가이드

| 개선 방향 | 내용 |
|---|---|
| **다중 피험자 학습** | `load_subject()` 반복 → 피험자 독립 모델 |
| **도메인 적응** | 피험자 간 도메인 시프트 해결 (DANN, MMD) |
| **데이터 증강** | 가우시안 노이즈, 시간 워핑, 채널 셔플 |
| **웨이블릿 특징** | `pywt.wavedec()` → CWT/DWT 계수 추가 |
| **Transformer** | 멀티헤드 어텐션으로 채널 간 관계 모델링 |
| **하이퍼파라미터 튜닝** | Optuna / Ray Tune 사용 |
| **교차 검증** | Leave-One-Subject-Out (LOSO-CV) |

```python
# 다중 피험자 예시
all_X, all_y, all_r, all_subj = [], [], [], []
for sid in range(1, 11):   # 피험자 1~10
    d = load_subject(sid)
    emg_f = preprocess_emg(d['emg'])
    X, y, r = segment_signal(emg_f, d['labels'], d['reps'])
    all_X.append(X);  all_y.append(y);  all_r.append(r)
    all_subj.extend([sid] * len(X))

X_pool = np.concatenate(all_X)
y_pool = np.concatenate(all_y)
```